In [4]:
from openai import OpenAI
from pydantic import BaseModel
import json
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()

class Step(BaseModel):
    explanation: str
    output: str

class MathReasoning(BaseModel):
    steps: list[Step]
    final_answer: str


def make_batch_line_from_sdk(model, messages, text_format_model, custom_id):
    """
    Generate a valid JSONL line for Batch API using OpenAI SDK internals.
    """
    # 1️⃣ Nhờ SDK tạo sẵn response_format JSON Schema
    schema = text_format_model.model_json_schema()

    # 2️⃣ Lấy JSON body đúng chuẩn mà SDK sẽ gửi
    body = {
        "model": model,
        "input": messages,
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": text_format_model.__name__,
                "schema": schema,
                "strict": True
            }
        }
    }

    # 3️⃣ Tạo dòng JSONL hợp lệ
    line = {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/responses",  # hoặc /v1/chat/completions nếu bạn dùng messages
        "body": body
    }

    return line


# Example usage
batch_line = make_batch_line_from_sdk(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "You are a math tutor."},
        {"role": "user", "content": "solve 8x + 7 = -23"}
    ],
    text_format_model=MathReasoning,
    custom_id="math-job-1"
)

# Ghi ra file JSONL
with open("batch_math.jsonl", "w", encoding="utf-8") as f:
    f.write(json.dumps(batch_line) + "\n")

print("✓ Created JSONL with valid schema from SDK model")


✓ Created JSONL with valid schema from SDK model


In [6]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {"role": "system", "content": "You are a helpful math tutor. Guide the user through the solution step by step."},
        {"role": "user", "content": "how can I solve 8x + 7 = -23"},
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "math_response",
            "schema": {
                "type": "object",
                "properties": {
                    "steps": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "explanation": {"type": "string"},
                                "output": {"type": "string"},
                            },
                            "required": ["explanation", "output"],
                            "additionalProperties": False,
                        },
                    },
                    "final_answer": {"type": "string"},
                },
                "required": ["steps", "final_answer"],
                "additionalProperties": False,
            },
            "strict": True,
        }
    }
)

print(response.output_text)


{"steps":[{"explanation":"The equation is 8x + 7 = -23. To solve for x, first subtract 7 from both sides to isolate the term with x.","output":"8x + 7 - 7 = -23 - 7, which simplifies to 8x = -30."},{"explanation":"Now divide both sides of the equation by 8 to solve for x.","output":"8x / 8 = -30 / 8, which simplifies to x = -30/8."},{"explanation":"Simplify the fraction -30/8 by dividing numerator and denominator by their greatest common divisor, which is 2.","output":"x = -15/4."}],"final_answer":"x = -15/4"}


In [11]:
import json
import requests
import os

# ========= 1️⃣ Đọc file JSONL kết quả batch =========
jsonl_path = "./test_single_window/batch_review_results.jsonl"

with open(jsonl_path, "r", encoding="utf-8") as f:
    data = json.loads(f.readline().strip())

# ========= 2️⃣ Lấy response_id, container_id và danh sách file =========
response_id = data["response"]["body"]["id"]

# Tìm container_id từ phần code_interpreter_call
container_output = next(
    (o for o in data["response"]["body"]["output"] if o.get("type") == "code_interpreter_call"),
    None
)
container_id = container_output["container_id"] if container_output else None

annotations = data["response"]["body"]["output"][-1]["content"][0].get("annotations", [])

# ========= 3️⃣ Lấy API key =========
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("❌ Thiếu OPENAI_API_KEY trong biến môi trường!")

headers = {"Authorization": f"Bearer {api_key}"}

# ========= 4️⃣ Helper: thử tải với nhiều endpoint =========
def fetch_file(file_id: str, filename: str):
    """
    Cố gắng tải file qua các endpoint khác nhau.
    Thứ tự thử:
      1. /responses/{response_id}/files/{file_id}/content
      2. /containers/{container_id}/files/{file_id}/content
      3. /files/{file_id}/content
    """
    urls = []
    if response_id:
        urls.append(f"https://api.openai.com/v1/responses/{response_id}/files/{file_id}/content")
    if container_id:
        urls.append(f"https://api.openai.com/v1/containers/{container_id}/files/{file_id}/content")
    urls.append(f"https://api.openai.com/v1/files/{file_id}/content")

    for url in urls:
        print(f"   → Thử tải từ {url}")
        r = requests.get(url, headers=headers)
        if r.status_code == 200:
            with open(filename, "wb") as f:
                f.write(r.content)
            print(f"✅  Đã lưu {filename} ({len(r.content)/1024:.1f} KB)")
            return True
        else:
            print(f"   ⚠️  {r.status_code} - {r.text[:120]}...")
    print(f"❌ Không tải được {filename} từ bất kỳ endpoint nào.\n")
    return False


# ========= 5️⃣ Tải từng file =========
if not annotations:
    print("⚠️ Không tìm thấy annotations trong JSON.")
else:
    for ann in annotations:
        file_id = ann["file_id"]
        filename = ann["filename"]
        print(f"\n⬇️ Đang tải {filename} (file_id={file_id}) ...")
        fetch_file(file_id, filename)



⬇️ Đang tải cfile_690c4fe6e8148191b968290b6af22738.png (file_id=cfile_690c4fe6e8148191b968290b6af22738) ...
   → Thử tải từ https://api.openai.com/v1/responses/resp_03f8b0a907634a4e00690c4f9bdc1881949be567b398faf9a9/files/cfile_690c4fe6e8148191b968290b6af22738/content
   ⚠️  404 - {
  "error": {
    "message": "Invalid URL (GET /v1/responses/resp_03f8b0a907634a4e00690c4f9bdc1881949be567b398faf9a9/fi...
   → Thử tải từ https://api.openai.com/v1/containers/cntr_690c4f9ead4c81908e0e6ee46dfd221e091d74ba6e7b4b39/files/cfile_690c4fe6e8148191b968290b6af22738/content
   ⚠️  404 - {
  "error": {
    "message": "Container is expired.",
    "type": "invalid_request_error",
    "param": null,
    "code...
   → Thử tải từ https://api.openai.com/v1/files/cfile_690c4fe6e8148191b968290b6af22738/content
   ⚠️  404 - {
  "error": {
    "message": "No such File object: cfile_690c4fe6e8148191b968290b6af22738",
    "type": "invalid_reques...
❌ Không tải được cfile_690c4fe6e8148191b968290b6af22738.png từ b

In [28]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()

class Calculator(BaseModel):
    formula: str
    result: float

response = client.responses.parse(
    model="gpt-4.1-mini",
    input=[
        {"role": "system", "content": "You have to use code tool for calculation"},
        {
            "role": "user",
            "content": "use code interpreter to calculate the nearest day have 366 days if today is 2025",
        },
    ],
    tools =[
        {
            "type": "code_interpreter",
            "container": {"type": "auto"}
        }
    ],
    text_format=Calculator,
)

event = response.output_parsed

In [ ]:
print(response)

In [30]:
print(event)

formula='Find the nearest leap year to 2025 by checking each subsequent year for leap year status.' result=2028.0
